# Extração do catálogo — Drogaria Araujo (v2)

Versão reescrita depois do `TargetClosedError`. Três mudanças de fundo:

| | versão anterior | esta versão |
|---|---|---|
| como pagina | clica em "próxima página" e lê o DOM renderizado | chama direto o endpoint de grid do Salesforce (`Search-UpdateGrid`) |
| itens por requisição | 48 | 500 |
| paralelismo | 1 página por vez | 6 blocos simultâneos |
| se cair | perde tudo, recomeça do zero | retoma de onde parou (`progresso-araujo.json`) |

Velocidade medida: **~250 produtos/s** contra os 3,78/s da versão anterior.

## Por que deu erro

```
TargetClosedError: Page.wait_for_timeout: Target page, context or browser has been closed
```

A janela do Chromium morreu no meio da raspagem (categoria Dermocosméticos, página 123, depois de ~40 min).
Como o navegador some, todo comando seguinte do Playwright estoura. Causas típicas: a janela foi fechada
no braço, o Mac dormiu, ou o Chromium travou depois de horas com páginas gigantes de 3,4 MB abertas.

O erro em si não é o problema principal — **o problema é que 40 minutos de trabalho foram perdidos**,
porque o script recriava o CSV do zero (`mode="w"`) a cada execução e não guardava em que ponto estava.

Outros dois defeitos que estavam ali:

1. **Dois slugs errados**: `saude-e-bem-estar` e `beleza-e-cuidados` não existem — o site usa `/saude` e
   `/beleza`. Essas duas categorias (128 e 37 subcategorias) voltariam vazias.
2. **Atributo errado**: o código lia `data-gtm4data`, mas no HTML o atributo é `data-gtmga4data`.
   Por isso o preço vinha do regex no texto do card, e não do JSON estruturado.

## 1. Setup

In [1]:
%pip install playwright pandas beautifulsoup4 tqdm -q
!python -m playwright install chromium -q

Note: you may need to restart the kernel to use updated packages.


zsh:1: command not found: python


In [2]:
import asyncio
import csv
import json
import os
import random
import re
import time
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
from tqdm.auto import tqdm

BASE = "https://www.araujo.com.br"
GRID = BASE + "/on/demandware.store/Sites-Araujo-Site/pt_BR/Search-UpdateGrid?cgid={cgid}&start={start}&sz={sz}"

UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36")

TAMANHO_BLOCO = 500      # produtos por requisição (o site aceita até 500)
CONCORRENCIA  = 6        # blocos baixados ao mesmo tempo
TENTATIVAS    = 4        # tentativas por bloco antes de desistir

CSV_PRODUTOS  = Path("produtos-araujo.csv")
CSV_CATEGORIAS = Path("categorias-araujo.csv")   # gerado no notebook anterior
ARQ_PROGRESSO = Path("progresso-araujo.json")

COLUNAS = ["departamento", "categoria", "subcategoria", "pid", "produto", "marca",
           "preco", "preco_de", "desconto", "url", "imagem"]

# recursos que não interessam: cortar isso deixa o carregamento ~5x mais rápido
RECURSOS_PESADOS = {"image", "media", "font", "stylesheet"}

## 2. Retomada

O estado vive em dois arquivos:

- `produtos-araujo.csv` — aberto em modo **append**, nunca recriado
- `progresso-araujo.json` — quais blocos (`cgid|start`) já foram gravados e onde cada departamento acaba

Se o navegador morrer de novo, é só rodar a célula de execução outra vez: os blocos já feitos são pulados.

In [3]:
def carregar_progresso() -> dict:
    if ARQ_PROGRESSO.exists():
        return json.loads(ARQ_PROGRESSO.read_text(encoding="utf-8"))
    return {"blocos_ok": [], "fim": {}, "falhas": []}


def salvar_progresso(estado: dict) -> None:
    ARQ_PROGRESSO.write_text(
        json.dumps(estado, ensure_ascii=False, indent=1), encoding="utf-8"
    )


def preparar_csv() -> set:
    """Cria o CSV se não existir e devolve as chaves (departamento, pid) já gravadas."""
    if not CSV_PRODUTOS.exists():
        with open(CSV_PRODUTOS, "w", newline="", encoding="utf-8-sig") as f:
            csv.DictWriter(f, fieldnames=COLUNAS, delimiter=";").writeheader()
        return set()

    ja_tem = set()
    with open(CSV_PRODUTOS, newline="", encoding="utf-8-sig") as f:
        for linha in csv.DictReader(f, delimiter=";"):
            ja_tem.add((linha["departamento"], linha["pid"]))
    return ja_tem


def gravar(linhas: list) -> None:
    if not linhas:
        return
    with open(CSV_PRODUTOS, "a", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=COLUNAS, delimiter=";")
        w.writerows(linhas)
        f.flush()
        os.fsync(f.fileno())

## 3. Parser do card de produto

Cada card traz um JSON do Google Analytics (`data-gtmga4data`) com nome, marca, preço de tabela e
**a categoria e a subcategoria do produto** — ou seja, a árvore de 750 categorias sai de graça,
sem precisar visitar cada uma. O preço com desconto vem do HTML (`.productPrice__price`).

In [4]:
def moeda(txt):
    """'R$1.314,38' -> 1314.38"""
    if not txt:
        return None
    m = re.search(r"([\d.]+,\d{2})", txt)
    return float(m.group(1).replace(".", "").replace(",", ".")) if m else None


def extrair_produtos(html: str, departamento: str) -> list:
    sopa = BeautifulSoup(html, "html.parser")
    linhas = []

    for card in sopa.select("div.productTile.js-product-tile"):
        dados = {}
        marcador = card.select_one("[data-gtmga4data]")
        if marcador:
            try:
                dados = json.loads(marcador["data-gtmga4data"])
            except (json.JSONDecodeError, KeyError):
                pass

        nome = card.get("title") or dados.get("item_name")
        href = card.get("data-url", "")
        atual = card.select_one(".productPrice__price")
        de = card.select_one(".productPrice__lineThrough")
        tag = card.select_one(".productTile__tag")
        img = card.select_one("img")

        linhas.append({
            "departamento": departamento,
            "categoria":    dados.get("item_category"),
            "subcategoria": dados.get("item_category2"),
            "pid":          card.get("data-pid"),
            "produto":      (nome or "").strip(),
            "marca":        dados.get("item_brand"),
            "preco":        moeda(atual.get_text() if atual else None),
            "preco_de":     moeda(de.get_text() if de else None) or dados.get("price"),
            "desconto":     tag.get_text(strip=True) if tag else None,
            "url":          BASE + href if href.startswith("/") else href,
            "imagem":       (img.get("data-src") or img.get("src")) if img else None,
        })

    return linhas

## 4. Descobrir o `cgid` de cada departamento

O endpoint de grid não aceita o slug (`/medicamentos`), só o id numérico interno (`cgid=1`).
O id aparece no HTML da página da categoria, no link de "próxima página"
(`/busca?cgid=1&start=48&sz=48&page=2`).

In [5]:
async def descobrir_cgids(ctx, departamentos: list) -> dict:
    encontrados = {}

    async def um(dep):
        for _ in range(TENTATIVAS):
            try:
                r = await ctx.request.get(dep["url"], timeout=90000)
                ids = re.findall(r"cgid=(\d+)", await r.text())
                if ids:
                    # o id do próprio departamento é o que mais se repete
                    encontrados[dep["categoria"]] = max(set(ids), key=ids.count)
                    return
            except Exception:
                await asyncio.sleep(2)

    await asyncio.gather(*[um(d) for d in departamentos])
    return encontrados

## 5. O motor

Um bloco = uma requisição de 500 produtos. Os blocos de um departamento são baixados em levas de
`CONCORRENCIA`; quando um bloco volta com menos de 500 produtos, chegou ao fim daquele departamento.

Bloco que falhar 4 vezes vai para `falhas` no progresso e não trava o resto — na próxima execução
ele é tentado de novo, porque não entrou em `blocos_ok`.

In [6]:
async def baixar_bloco(ctx, cgid, start, barra):
    """Devolve o HTML do bloco, com tentativas e espera progressiva."""
    url = GRID.format(cgid=cgid, start=start, sz=TAMANHO_BLOCO)

    for tentativa in range(TENTATIVAS):
        try:
            r = await ctx.request.get(url, timeout=120000)
            if r.status == 200:
                return await r.text()
            barra.write(f"  bloco {cgid}|{start}: HTTP {r.status}")
        except Exception as e:
            barra.write(f"  bloco {cgid}|{start} tentativa {tentativa + 1}: {str(e)[:70]}")
        await asyncio.sleep(2 ** tentativa + random.random())

    return None


async def raspar_departamento(ctx, nome, cgid, estado, ja_tem, barra):
    feitos = {int(b.split("|")[1]) for b in estado["blocos_ok"] if b.startswith(f"{cgid}|")}
    fim = estado["fim"].get(str(cgid))
    inicio = 0
    total_dep = 0

    while True:
        leva = [inicio + i * TAMANHO_BLOCO for i in range(CONCORRENCIA)]
        leva = [s for s in leva if s not in feitos and (fim is None or s <= fim)]

        if not leva:
            if fim is not None and all(s in feitos for s in range(0, fim + 1, TAMANHO_BLOCO)):
                break
            inicio += TAMANHO_BLOCO * CONCORRENCIA
            if fim is not None and inicio > fim:
                break
            continue

        htmls = await asyncio.gather(*[baixar_bloco(ctx, cgid, s, barra) for s in leva])

        for start, html in zip(leva, htmls):
            if html is None:
                estado["falhas"].append(f"{cgid}|{start}")
                continue

            produtos = extrair_produtos(html, nome)

            novos = [p for p in produtos if (nome, p["pid"]) not in ja_tem]
            for p in novos:
                ja_tem.add((nome, p["pid"]))
            gravar(novos)

            feitos.add(start)
            estado["blocos_ok"].append(f"{cgid}|{start}")
            total_dep += len(novos)
            barra.update(len(novos))
            barra.set_postfix({"dep": nome[:14], "start": start})

            if len(produtos) < TAMANHO_BLOCO:
                fim = start if fim is None else min(fim, start)
                estado["fim"][str(cgid)] = fim

        salvar_progresso(estado)

        if fim is not None and all(s in feitos for s in range(0, fim + 1, TAMANHO_BLOCO)):
            break
        inicio += TAMANHO_BLOCO * CONCORRENCIA

    return total_dep

In [7]:
async def raspar_tudo():
    estado = carregar_progresso()
    ja_tem = preparar_csv()
    print(f"{len(ja_tem)} produtos já no CSV | {len(estado['blocos_ok'])} blocos já baixados")

    departamentos = (pd.read_csv(CSV_CATEGORIAS)
                       .query("nivel == 1")[["categoria", "url"]]
                       .to_dict("records"))

    inicio = time.time()

    async with async_playwright() as p:
        navegador = await p.chromium.launch(headless=False, args=["--disable-http2"])
        ctx = await navegador.new_context(
            user_agent=UA, locale="pt-BR", timezone_id="America/Sao_Paulo",
            viewport={"width": 1440, "height": 900},
        )

        # a primeira visita é o que gera os cookies que passam pelo WAF da Akamai
        pagina = await ctx.new_page()

        # não baixa imagem/fonte/css: só interessa o HTML.
        # ATENÇÃO: tem que ser pagina.route e não ctx.route -- no contexto o filtro também
        # intercepta as chamadas de ctx.request.get e elas passam a estourar timeout.
        async def filtrar(rota):
            if rota.request.resource_type in RECURSOS_PESADOS:
                await rota.abort()
            else:
                await rota.continue_()

        await pagina.route("**/*", filtrar)

        await pagina.goto(BASE, wait_until="domcontentloaded", timeout=90000)
        await pagina.wait_for_timeout(2500)

        cgids = await descobrir_cgids(ctx, departamentos)
        print("cgids:", cgids)

        barra = tqdm(desc="Produtos", unit=" prod", initial=len(ja_tem))
        resumo = {}

        for dep in departamentos:
            nome = dep["categoria"]
            cgid = cgids.get(nome)
            if not cgid:
                barra.write(f"!! cgid não encontrado para {nome}, pulando")
                continue
            resumo[nome] = await raspar_departamento(ctx, nome, cgid, estado, ja_tem, barra)

        barra.close()
        await navegador.close()

    salvar_progresso(estado)
    minutos = (time.time() - inicio) / 60

    print(f"\n=========== RESUMO ({minutos:.1f} min) ===========")
    for nome, qtd in resumo.items():
        print(f"  {nome:20} {qtd:6d} novos")
    print(f"  {'TOTAL no CSV':20} {len(ja_tem):6d}")
    if estado["falhas"]:
        print(f"  blocos com falha: {len(estado['falhas'])} — rode a célula de novo para recuperá-los")

## 6. Rodar

Pode interromper (botão de parar / fechar o Chromium) e rodar de novo: continua de onde parou.
Para começar do zero, apague `produtos-araujo.csv` e `progresso-araujo.json`.

In [8]:
await raspar_tudo()

0 produtos já no CSV | 0 blocos já baixados


cgids: {'Beleza e Cuidados': '2', 'Higiene Pessoal': '3', 'Mercado': '5', 'Dermocosméticos': '199', 'Cabelo': '321', 'Saúde e Bem Estar': '10', 'Pet Shop': '6', 'Medicamentos': '1', 'Maquiagem': '186', 'Nutrição Saudável': '9'}


Produtos: 0 prod [00:00, ? prod/s]

!! cgid não encontrado para Infantil, pulando



=========== RESUMO (3.0 min) ===========
  Medicamentos           7901 novos
  Dermocosméticos        1527 novos
  Saúde e Bem Estar      2426 novos
  Beleza e Cuidados      2162 novos
  Higiene Pessoal        2074 novos
  Pet Shop                705 novos
  Nutrição Saudável      1258 novos
  Mercado                3423 novos
  Maquiagem               756 novos
  Cabelo                 2484 novos
  TOTAL no CSV          24716


## 7. Conferindo o resultado

In [9]:
df = pd.read_csv(CSV_PRODUTOS, sep=";", encoding="utf-8-sig")

print(f"linhas no CSV:        {len(df):,}")
print(f"produtos distintos:   {df['pid'].nunique():,}")
print(f"marcas:               {df['marca'].nunique():,}")
print(f"subcategorias:        {df['subcategoria'].nunique():,}")
print(f"sem preço:            {df['preco'].isna().sum():,}")
df.head()

linhas no CSV:        24,716
produtos distintos:   24,716
marcas:               4,548
subcategorias:        107
sem preço:            1,125


,departamento,categoria,subcategoria,pid,produto,marca,preco,preco_de,desconto,url,imagem
0,Medicamentos,Medicamentos,Remédio para Emagrecer,101392,Wegovy 1mg Semaglutida com 4 Doses Injetáveis,Wegovy,999.0,1314.38,"23,99% OFF",https://www.araujo.com.br/wegovy-1mg-semagluti...,https://i3-imagens-prd.araujo.com.br/redimensi...
1,Medicamentos,Medicamentos,Remédio para Emagrecer,101352,"Wegovy 0,5mg Semaglutida com 4 Doses Injetáveis",Wegovy,975.0,1314.38,"25,82% OFF",https://www.araujo.com.br/wegovy-05mg-semaglut...,https://i3-imagens-prd.araujo.com.br/redimensi...
2,Medicamentos,Medicamentos,Remédio para Emagrecer,101394,"Wegovy 2,4mg Semaglutida com 4 Doses Injetáveis",Wegovy,1749.0,2532.31,"30,93% OFF",https://www.araujo.com.br/wegovy-24mg-semaglut...,https://i3-imagens-prd.araujo.com.br/redimensi...
3,Medicamentos,Medicamentos,Remédio para Emagrecer,101393,"Wegovy 1,7mg Semaglutida com 4 Doses Injetáveis",Wegovy,1399.0,1968.82,"28,94% OFF",https://www.araujo.com.br/wegovy-17mg-semaglut...,https://i3-imagens-prd.araujo.com.br/redimensi...
4,Medicamentos,Medicamentos,Remédio para Emagrecer,101395,"Wegovy 0,25mg Semaglutida com 4 Doses Injetáveis",Wegovy,899.0,1314.38,"31,6% OFF",https://www.araujo.com.br/wegovy-025mg-semaglu...,https://i3-imagens-prd.araujo.com.br/redimensi...


In [10]:
(df.groupby("departamento")
   .agg(produtos=("pid", "nunique"),
        preco_medio=("preco", "mean"),
        preco_max=("preco", "max"))
   .sort_values("produtos", ascending=False)
   .round(2))

,produtos,preco_medio,preco_max
departamento,,,
Medicamentos,7901,714.24,139040.89
Mercado,3423,16.03,353.69
Cabelo,2484,37.32,313.49
Saúde e Bem Estar,2426,85.32,2349.99
Beleza e Cuidados,2162,38.98,499.99
Higiene Pessoal,2074,24.90,899.99
Dermocosméticos,1527,119.14,878.79
Nutrição Saudável,1258,57.16,569.99
Maquiagem,756,37.95,204.29


In [11]:
# um produto pode estar em mais de um departamento; esta é a visão por produto único
catalogo = (df.sort_values("preco")
              .drop_duplicates(subset="pid", keep="first")
              .reset_index(drop=True))

catalogo.to_csv("catalogo-araujo-unico.csv", index=False, sep=";", encoding="utf-8-sig")
print(f"{len(catalogo):,} produtos únicos salvos em catalogo-araujo-unico.csv")
catalogo["preco"].describe().round(2)

24,716 produtos únicos salvos em catalogo-araujo-unico.csv


count     23591.00
mean        262.94
std        2718.57
min           0.01
25%          14.69
50%          34.59
75%          78.99
max      139040.89
Name: preco, dtype: float64